# 🔍 RAG From Scratch — Technical Datasheet Q&A System
### ECE Department | Manipal Institute of Technology
**Stack:** Groq (LLaMA 3) · Sentence-Transformers · ChromaDB · LangChain  
**Parts covered:** 1–4 (Basic RAG) · 5 (Multi-Query) · 6 (RAG-Fusion/RRF) · 9 (HyDE)

---
> **How to use this notebook:**
> 1. Get a **free Groq API key** at [console.groq.com](https://console.groq.com) (1 minute, no card needed)
> 2. Place your **datasheet PDFs** in the `datasheets/` folder next to this notebook
> 3. Edit the **CONFIG cell** (Step 2) with your API key, PDF filenames, and questions
> 4. Run all cells: `Kernel → Restart & Run All`


## 📦 Step 1: Install Dependencies
*Run once. ~2 minutes on first run.*

In [1]:
!pip install -q langchain langchain-community langchain-groq langchain-huggingface
!pip install -q sentence-transformers chromadb
!pip install -q pypdf tiktoken gradio matplotlib
print("✅ All packages installed.")


✅ All packages installed.


## ⚙️ Step 2: Configuration — Edit This Cell
> This is the **only cell you need to edit** before running.


In [3]:
# ================================================================
#  USER CONFIGURATION
# ================================================================
import os

# 1. FREE GROQ API KEY -- get it at https://console.groq.com
GROQ_API_KEY = "your_groq_api_key_here"   # <- paste your key here
os.environ["GROQ_API_KEY"] = GROQ_API_KEY

# 2. PDF PATHS -- list your datasheet PDFs here
PDF_PATHS = [
    "datasheets/stm32f103_datasheet.pdf",  # <- replace with your file paths
    "datasheets/ESP32_datasheet.pdf",
]
# To load all PDFs in a folder, uncomment below:
# import glob; PDF_PATHS = glob.glob('datasheets/*.pdf')

# 3. YOUR QUESTIONS -- ask anything about the datasheets
QUESTIONS = [
    "What is the maximum operating voltage of this microcontroller?",
    "List all communication interfaces: UART, SPI, I2C, USB.",
    "What is the flash memory capacity and SRAM size?",
    "What are the GPIO pin current limits?",
    "Describe power consumption in sleep or standby mode.",
]

# 4. MODEL SETTINGS (no need to change)
LLM_MODEL   = "llama3-8b-8192"       # free Groq model
EMBED_MODEL = "all-MiniLM-L6-v2"     # free local embeddings
CHUNK_SIZE  = 500
CHUNK_OVERLAP = 100
TOP_K       = 4

print("Configuration loaded.")
print(f"  LLM       : {LLM_MODEL} via Groq (free)")
print(f"  Embeddings: {EMBED_MODEL} (local, free)")
print(f"  PDFs      : {len(PDF_PATHS)} listed")
print(f"  Questions : {len(QUESTIONS)}")


Configuration loaded.
  LLM       : llama3-8b-8192 via Groq (free)
  Embeddings: all-MiniLM-L6-v2 (local, free)
  PDFs      : 2 listed
  Questions : 5


## 📄 Step 3: Load Datasheet PDFs
Loads PDFs page by page. If no PDFs are found, a built-in demo STM32 datasheet is used so the notebook still runs end-to-end.


In [4]:
from langchain_community.document_loaders import PyPDFLoader
from langchain.schema import Document
from pathlib import Path

all_docs = []
for pdf_path in PDF_PATHS:
    p = Path(pdf_path)
    if not p.exists():
        print(f"  WARNING: {pdf_path} not found, skipping.")
        continue
    docs = PyPDFLoader(str(p)).load()
    all_docs.extend(docs)
    print(f"  Loaded {p.name} -> {len(docs)} pages")

if not all_docs:
    print("No PDFs found. Using built-in STM32 demo datasheet.")
    demo = (
        "STM32F103 Microcontroller Technical Datasheet\n\n"
        "OVERVIEW\n"
        "ARM Cortex-M3 core, up to 72 MHz clock speed.\n\n"
        "POWER SUPPLY\n"
        "Operating voltage: 2.0V to 3.6V.\n"
        "I/O pins are 5V tolerant.\n"
        "Stop mode current: 20 uA at 25C.\n"
        "Standby mode current: 2 uA at 25C.\n\n"
        "MEMORY\n"
        "Flash: 64 KB to 512 KB depending on variant.\n"
        "SRAM: 20 KB.\n\n"
        "GPIO\n"
        "Up to 51 GPIO pins.\n"
        "Max sink/source current per pin: 25 mA.\n"
        "Total I/O current: 150 mA.\n\n"
        "COMMUNICATION INTERFACES\n"
        "USART: 3 interfaces, up to 4.5 Mbit/s.\n"
        "SPI: 2 interfaces, up to 18 Mbit/s.\n"
        "I2C: 2 interfaces, up to 400 kHz.\n"
        "USB: Full-speed 12 Mbit/s USB 2.0.\n"
        "CAN: 1 x CAN 2.0B.\n\n"
        "ADC\n"
        "2 x 12-bit ADC, up to 16 channels.\n"
        "Conversion time: 1 microsecond.\n"
        "Input range: 0 to 3.6V.\n\n"
        "TEMPERATURE RANGE\n"
        "Industrial: -40C to +85C.\n"
        "Packages: LQFP48, LQFP64, LQFP100.\n"
    )
    all_docs = [Document(page_content=demo, metadata={"source": "DEMO_STM32.pdf", "page": 0})]

print(f"\nTotal pages loaded: {len(all_docs)}")
print("\nPreview of first page (first 300 chars):")
print(all_docs[0].page_content[:300])


No PDFs found. Using built-in STM32 demo datasheet.

Total pages loaded: 1

Preview of first page (first 300 chars):
STM32F103 Microcontroller Technical Datasheet

OVERVIEW
ARM Cortex-M3 core, up to 72 MHz clock speed.

POWER SUPPLY
Operating voltage: 2.0V to 3.6V.
I/O pins are 5V tolerant.
Stop mode current: 20 uA at 25C.
Standby mode current: 2 uA at 25C.

MEMORY
Flash: 64 KB to 512 KB depending on variant.
SRAM


## ✂️ Step 4: Chunk Documents
Splits pages into overlapping segments. Smaller chunks improve retrieval precision.


In [5]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\n\n", "\n", " ", ""],
)
chunks = splitter.split_documents(all_docs)

print(f"Original pages : {len(all_docs)}")
print(f"Chunks created : {len(chunks)}")
print(f"Avg chunk size : {int(sum(len(c.page_content) for c in chunks)/len(chunks))} chars")
print("\nSample chunk:")
print(chunks[0].page_content[:250])


Original pages : 1
Chunks created : 2
Avg chunk size : 428 chars

Sample chunk:
STM32F103 Microcontroller Technical Datasheet

OVERVIEW
ARM Cortex-M3 core, up to 72 MHz clock speed.

POWER SUPPLY
Operating voltage: 2.0V to 3.6V.
I/O pins are 5V tolerant.
Stop mode current: 20 uA at 25C.
Standby mode current: 2 uA at 25C.

MEMORY


## 🧠 Step 5: Embed & Build Vector Index
`all-MiniLM-L6-v2` runs **locally** on CPU — completely free, no API needed.  
Downloads ~80 MB on first run, cached after that.


In [7]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

print("Loading embedding model (downloads ~80MB on first run)...")
embedding_model = HuggingFaceEmbeddings(
    model_name=EMBED_MODEL,
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)
print("Embedding model ready.")

print("Indexing chunks into ChromaDB...")
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    collection_name="datasheet_rag",
)
retriever = vectorstore.as_retriever(search_kwargs={"k": TOP_K})
print(f"Vector index built with {len(chunks)} chunks.")

# Sanity check
test = retriever.invoke("operating voltage")
print(f"\nSanity check for 'operating voltage' -> {len(test)} docs retrieved.")
print(test[0].page_content[:200])


Loading embedding model (downloads ~80MB on first run)...


C:\Users\bpssi\anaconda3\Lib\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


Embedding model ready.
Indexing chunks into ChromaDB...
Vector index built with 2 chunks.

Sanity check for 'operating voltage' -> 4 docs retrieved.
STM32F103 Microcontroller Technical Datasheet

OVERVIEW
ARM Cortex-M3 core, up to 72 MHz clock speed.

POWER SUPPLY
Operating voltage: 2.0V to 3.6V.
I/O pins are 5V tolerant.
Stop mode current: 20 uA 


## 🤖 Step 6: Connect Groq LLM
[Groq](https://console.groq.com) provides **free inference** for LLaMA 3, Gemma, Mixtral.  
Sign up, copy your API key, paste it in the config cell above.


In [8]:
from langchain_groq import ChatGroq
from langchain_core.output_parsers import StrOutputParser
from langchain.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

llm = ChatGroq(model=LLM_MODEL, temperature=0, groq_api_key=GROQ_API_KEY)

RAG_TEMPLATE = (
    "You are a technical assistant helping engineers understand electronic datasheets.\n"
    "Answer ONLY using the context below. Be precise: include numbers, units, and values.\n"
    "If the answer is not in the context, say: 'Not found in the provided datasheet.'\n\n"
    "Context:\n{context}\n\n"
    "Question: {question}\n\n"
    "Answer:"
)
rag_prompt = ChatPromptTemplate.from_template(RAG_TEMPLATE)

def format_docs(docs):
    parts = []
    for d in docs:
        src = d.metadata.get("source", "?")
        pg  = d.metadata.get("page", "?")
        parts.append(f"[{src} | p.{pg}]\n{d.page_content}")
    return "\n\n---\n\n".join(parts)

# Basic RAG chain
basic_rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt | llm | StrOutputParser()
)

# Quick test
print("Testing LLM connection...")
ans = basic_rag_chain.invoke(QUESTIONS[0])
print(f"Q: {QUESTIONS[0]}")
print(f"A: {ans}")
print("\nLLM connected successfully.")


Testing LLM connection...


AuthenticationError: Error code: 401 - {'error': {'message': 'Invalid API Key', 'type': 'invalid_request_error', 'code': 'invalid_api_key'}}

## 🔷 Method 1: Basic RAG (Parts 1–4)
**Pipeline:** User query → retrieve top-K chunks → prompt + LLM → answer.


In [ ]:
print("=" * 60)
print("  METHOD 1: BASIC RAG")
print("=" * 60)
basic_results = {}
for i, q in enumerate(QUESTIONS, 1):
    print(f"\nQ{i}: {q}")
    print("-" * 50)
    docs = retriever.invoke(q)
    ans  = basic_rag_chain.invoke(q)
    basic_results[q] = {"answer": ans, "num_docs": len(docs)}
    print(f"Docs retrieved : {len(docs)}")
    print(f"Answer         : {ans}")
print("\nBasic RAG done.")


## 🔷 Method 2: Multi-Query RAG (Part 5)
**Idea:** Generate 5 different phrasings of the question, retrieve for each, merge unique results.  
Recovers relevant chunks that a single phrasing would miss.


In [ ]:
from langchain.load import dumps, loads

MQ_TEMPLATE = (
    "Generate 5 different versions of this question to search a technical datasheet.\n"
    "Return ONLY the 5 questions, one per line, no numbering.\n\n"
    "Original: {question}"
)
mq_prompt = ChatPromptTemplate.from_template(MQ_TEMPLATE)

generate_queries = (
    mq_prompt
    | ChatGroq(model=LLM_MODEL, temperature=0.3, groq_api_key=GROQ_API_KEY)
    | StrOutputParser()
    | (lambda x: [l.strip() for l in x.strip().splitlines() if l.strip()])
)

def unique_union(docs_list):
    seen = {}
    for docs in docs_list:
        for doc in docs:
            key = dumps(doc)
            if key not in seen:
                seen[key] = doc
    return list(seen.values())

mq_retrieval = generate_queries | retriever.map() | unique_union
mq_chain = (
    {"context": mq_retrieval | format_docs, "question": RunnablePassthrough()}
    | rag_prompt | llm | StrOutputParser()
)

print("=" * 60)
print("  METHOD 2: MULTI-QUERY RAG")
print("=" * 60)
multi_results = {}
for i, q in enumerate(QUESTIONS, 1):
    print(f"\nQ{i}: {q}")
    print("-" * 50)
    sub_qs = generate_queries.invoke({"question": q})
    print(f"Generated queries:")
    for sq in sub_qs[:3]:
        print(f"  - {sq}")
    all_d = mq_retrieval.invoke({"question": q})
    ans   = mq_chain.invoke(q)
    multi_results[q] = {"answer": ans, "num_docs": len(all_d), "sub_queries": sub_qs}
    print(f"Unique docs    : {len(all_d)}")
    print(f"Answer         : {ans}")
print("\nMulti-Query done.")


## 🔷 Method 3: RAG-Fusion with RRF (Part 6)
**Idea:** Generate 4 related queries → retrieve for each → re-rank with Reciprocal Rank Fusion.  
Documents appearing consistently across multiple retrievals rank highest.

$$\text{RRF}(d) = \sum_i \frac{1}{k + \text{rank}_i(d)}, \quad k=60$$

In [ ]:
RF_TEMPLATE = (
    "Generate 4 search queries related to this datasheet question.\n"
    "Each should focus on a different technical aspect.\n"
    "Return ONLY the queries, one per line.\n\n"
    "Question: {question}"
)
rf_prompt = ChatPromptTemplate.from_template(RF_TEMPLATE)

gen_fusion_queries = (
    rf_prompt
    | ChatGroq(model=LLM_MODEL, temperature=0.3, groq_api_key=GROQ_API_KEY)
    | StrOutputParser()
    | (lambda x: [l.strip() for l in x.strip().splitlines() if l.strip()])
)

def reciprocal_rank_fusion(results, k=60):
    """RRF: score each doc across all ranked lists, return sorted by score."""
    scores = {}
    for docs in results:
        for rank, doc in enumerate(docs):
            key = dumps(doc)
            if key not in scores:
                scores[key] = {"doc": doc, "score": 0.0}
            scores[key]["score"] += 1.0 / (k + rank + 1)
    ranked = sorted(scores.values(), key=lambda x: x["score"], reverse=True)
    return [item["doc"] for item in ranked]

fusion_retrieval = gen_fusion_queries | retriever.map() | reciprocal_rank_fusion
fusion_chain = (
    {"context": fusion_retrieval | format_docs, "question": RunnablePassthrough()}
    | rag_prompt | llm | StrOutputParser()
)

print("=" * 60)
print("  METHOD 3: RAG-FUSION + RRF")
print("=" * 60)
fusion_results = {}
for i, q in enumerate(QUESTIONS, 1):
    print(f"\nQ{i}: {q}")
    print("-" * 50)
    fqs  = gen_fusion_queries.invoke({"question": q})
    print(f"Fusion queries:")
    for fq in fqs:
        print(f"  - {fq}")
    fd  = fusion_retrieval.invoke({"question": q})
    ans = fusion_chain.invoke(q)
    fusion_results[q] = {"answer": ans, "num_docs": len(fd)}
    print(f"RRF-ranked docs: {len(fd)}")
    print(f"Answer         : {ans}")
print("\nRAG-Fusion done.")


## 🔷 Method 4: HyDE — Hypothetical Document Embeddings (Part 9)
**Idea:** Instead of embedding the short question, generate a *hypothetical datasheet passage* that would answer it, then embed and retrieve using that passage.  
Bridges the gap between short query language and technical document language.


In [ ]:
HYDE_TEMPLATE = (
    "Write a short technical datasheet excerpt (2-4 sentences) that directly answers\n"
    "the question below. Use technical language with specific numbers and units.\n\n"
    "Question: {question}\n"
    "Datasheet excerpt:"
)
hyde_prompt = ChatPromptTemplate.from_template(HYDE_TEMPLATE)

gen_hyp_doc = (
    hyde_prompt
    | ChatGroq(model=LLM_MODEL, temperature=0.2, groq_api_key=GROQ_API_KEY)
    | StrOutputParser()
)

# Embed the hypothetical doc, retrieve real chunks with that embedding
hyde_retrieval = gen_hyp_doc | retriever
hyde_chain = (
    {"context": hyde_retrieval | format_docs, "question": RunnablePassthrough()}
    | rag_prompt | llm | StrOutputParser()
)

print("=" * 60)
print("  METHOD 4: HyDE")
print("=" * 60)
hyde_results = {}
for i, q in enumerate(QUESTIONS, 1):
    print(f"\nQ{i}: {q}")
    print("-" * 50)
    hyp = gen_hyp_doc.invoke({"question": q})
    print(f"Hypothetical doc: {hyp[:150]}...")
    hd  = hyde_retrieval.invoke({"question": q})
    ans = hyde_chain.invoke(q)
    hyde_results[q] = {"answer": ans, "num_docs": len(hd), "hyp_doc": hyp}
    print(f"Retrieved docs : {len(hd)}")
    print(f"Answer         : {ans}")
print("\nHyDE done.")


## 📊 Step 7: Side-by-Side Comparison

In [ ]:
from IPython.display import display, HTML

def make_table(q, b, m, f, h):
    rows = ""
    for label, res, bg, col in [
        ("Basic RAG",    b, "#EFF6FF", "#1E40AF"),
        ("Multi-Query",  m, "#F5F3FF", "#5B21B6"),
        ("RAG-Fusion",   f, "#FFF7ED", "#92400E"),
        ("HyDE",         h, "#F0FDF4", "#14532D"),
    ]:
        rows += (
            "<tr>"
            f"<td style='padding:8px 12px;font-weight:bold;color:{col};background:{bg};"
            f"white-space:nowrap;border-radius:4px'>{label}</td>"
            f"<td style='padding:8px 12px;font-size:13px;color:#374151'>{res['answer']}</td>"
            "</tr>"
        )
    return (
        "<div style='margin:14px 0;border:1px solid #E5E7EB;border-radius:8px;overflow:hidden'>"
        f"<div style='background:#111827;color:white;padding:10px 14px;font-weight:600;font-size:13px'>{q}</div>"
        "<table style='width:100%;border-collapse:collapse'>"
        "<thead><tr style='background:#F9FAFB'>"
        "<th style='padding:8px 12px;text-align:left;font-size:11px;color:#6B7280;width:120px'>Method</th>"
        "<th style='padding:8px 12px;text-align:left;font-size:11px;color:#6B7280'>Answer</th>"
        "</tr></thead>"
        f"<tbody>{rows}</tbody></table></div>"
    )

for q in QUESTIONS:
    display(HTML(make_table(q, basic_results[q], multi_results[q],
                            fusion_results[q], hyde_results[q])))


## 📈 Step 8: Retrieval Coverage Chart

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

labels   = [f'Q{i+1}' for i in range(len(QUESTIONS))]
methods  = ['Basic RAG', 'Multi-Query', 'RAG-Fusion', 'HyDE']
results  = [basic_results, multi_results, fusion_results, hyde_results]
colors   = ['#3B82F6', '#8B5CF6', '#F97316', '#10B981']

x, w = np.arange(len(QUESTIONS)), 0.2
fig, ax = plt.subplots(figsize=(11, 5))

for i, (method, res, col) in enumerate(zip(methods, results, colors)):
    vals = [res[q]['num_docs'] for q in QUESTIONS]
    bars = ax.bar(x + (i - 1.5)*w, vals, w, label=method,
                  color=col, alpha=0.85, edgecolor='white', linewidth=0.5)
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., h + 0.05,
                str(int(h)), ha='center', va='bottom', fontsize=9)

ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_xlabel('Question')
ax.set_ylabel('Documents Retrieved')
ax.set_title('Retrieval Coverage by Method', fontweight='bold', fontsize=13)
ax.legend(fontsize=10)
ax.spines[['top','right']].set_visible(False)
ax.set_facecolor('#FAFAFA')
fig.patch.set_facecolor('white')

for i, q in enumerate(QUESTIONS):
    plt.figtext(0.13, 0.04 - i*0.035, f'Q{i+1}: {q[:65]}',
                fontsize=8, color='#6B7280')

plt.tight_layout(rect=[0, 0.2, 1, 1])
plt.savefig('rag_coverage.png', dpi=150, bbox_inches='tight')
plt.show()
print("Chart saved as rag_coverage.png")


## 🖥️ Step 9: Interactive Gradio App
Run this cell → click the local URL → type any question → choose a RAG method.


In [ ]:
import gradio as gr

CHAINS = {
    "Basic RAG (Parts 1-4)":     basic_rag_chain,
    "Multi-Query RAG (Part 5)":  mq_chain,
    "RAG-Fusion / RRF (Part 6)": fusion_chain,
    "HyDE (Part 9)":             hyde_chain,
}

def ask(question, method):
    if not question.strip():
        return "Please enter a question."
    return CHAINS[method].invoke(question)

with gr.Blocks(title='Datasheet RAG', theme=gr.themes.Soft()) as demo:
    gr.Markdown(
        "# Technical Datasheet Q&A System\n"
        "**Manipal Institute of Technology | ECE Dept.**  \n"
        "Ask any question about the loaded datasheets."
    )
    with gr.Row():
        q_in  = gr.Textbox(label='Your Question',
                           placeholder="e.g. What is the maximum operating voltage?",
                           lines=2, scale=3)
        m_sel = gr.Dropdown(
            choices=list(CHAINS.keys()),
            value="Basic RAG (Parts 1-4)",
            label='RAG Method', scale=1
        )
    ans_out = gr.Textbox(label='Answer', lines=6, interactive=False)
    with gr.Row():
        gr.Button("Ask", variant="primary").click(ask, [q_in, m_sel], ans_out)
        gr.Button("Clear").click(lambda: ("", ""), outputs=[q_in, ans_out])
    gr.Examples(
        examples=[
            ["What is the maximum operating voltage?", "Basic RAG (Parts 1-4)"],
            ["List all communication interfaces.",     "Multi-Query RAG (Part 5)"],
            ["What is the flash memory size?",         "RAG-Fusion / RRF (Part 6)"],
            ["Describe GPIO current limits.",          "HyDE (Part 9)"],
        ],
        inputs=[q_in, m_sel]
    )

demo.launch(share=False)  # set share=True for a public URL


## ✅ Step 10: Project Summary

| Component | Technology | Cost |
|-----------|-----------|------|
| LLM | LLaMA 3 (8B) via Groq | Free |
| Embeddings | all-MiniLM-L6-v2 (local CPU) | Free |
| Vector DB | ChromaDB (in-memory) | Free |
| Orchestration | LangChain LCEL | Free |
| UI | Gradio | Free |

### RAG Methods Implemented
| Part | Method | Core Technique |
|------|--------|---------------|
| Parts 1–4 | Basic RAG | Retrieve → Prompt → Generate |
| Part 5 | Multi-Query | 5 variants → unique union |
| Part 6 | RAG-Fusion | 4 queries → RRF re-ranking |
| Part 9 | HyDE | Hypothetical doc → embed → retrieve |

### To customise
- **Different PDFs:** Update `PDF_PATHS` in the config cell
- **Different questions:** Update `QUESTIONS` in the config cell
- **Different LLM:** Change `LLM_MODEL` to `gemma2-9b-it` or `mixtral-8x7b-32768`
- **Persistent index:** Pass `persist_directory='./chroma_db'` to `Chroma.from_documents()`

---
*ECE Department · Manipal Institute of Technology · AI Project Submission*
